# 🏏 IPL Data Analysis (2008–2023)
**Author:** Dheeraj Kandpal | **GitHub:** [DheerajKandpal](https://github.com/DheerajKandpal)

**Dataset:** 988 matches · 5,018 batting records · 4,762 bowling records · 16 seasons

**Tools:** Python · pandas · matplotlib · seaborn

---

### 🎯 Objective
Full exploratory data analysis on IPL data covering:
- **Univariate:** Score distributions, title wins, toss decisions
- **Bivariate:** Team comparisons, batting vs bowling correlations
- **Trend analysis:** Season-over-season score inflation, strike rate evolution
- **Outlier detection:** Exceptional scores, dominant performances
- **Business insights:** Toss advantage, pace vs spin economics


## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# IPL dark theme
BG='#0D1117'; SRF='#161B22'; WHT='#E6EDF3'; MUT='#8B949E'
BLUE='#1E90FF'; ORG='#FF8C00'; RED='#FF4444'; GRN='#39D353'
YLW='#F0A500'; PURP='#B79FFF'
IPL_COLORS = {
    'Mumbai Indians':'#004BA0',
    'Chennai Super Kings':'#F5C000',
    'Royal Challengers Bangalore':'#EC1C24',
    'Kolkata Knight Riders':'#3A225D',
    'Delhi Capitals':'#0078BC',
    'Sunrisers Hyderabad':'#FF822A',
    'Rajasthan Royals':'#EA1A85',
    'Punjab Kings':'#ED1B24',
    'Gujarat Titans':'#1C1C1C',
    'Lucknow Super Giants':'#A0E0FF',
}
plt.rcParams.update({'figure.facecolor': BG, 'axes.facecolor': SRF, 'text.color': WHT})

# Load datasets
matches = pd.read_csv('../data/ipl_matches.csv')
batting = pd.read_csv('../data/ipl_batting.csv')
bowling = pd.read_csv('../data/ipl_bowling.csv')

print(f'Matches : {matches.shape}')
print(f'Batting : {batting.shape}')
print(f'Bowling : {bowling.shape}')

## 2. Dataset Overview & Quality Check

In [ ]:
print('=== MATCHES OVERVIEW ===')
print(f'Seasons     : {matches["season"].min()} – {matches["season"].max()}')
print(f'Total games : {len(matches):,}')
print(f'Teams       : {matches["team1"].nunique()}')
print(f'Venues      : {matches["venue"].nunique()}')
print(f'Null values : {matches.isnull().sum().sum()}')
print()
print('=== SCORE STATS ===')
all_scores = pd.concat([matches['team1_score'], matches['team2_score']])
print(f'Mean score  : {all_scores.mean():.1f}')
print(f'Median      : {all_scores.median():.1f}')
print(f'Std dev     : {all_scores.std():.1f}')
print(f'Highest     : {all_scores.max()}')
print(f'Lowest      : {all_scores.min()}')
print()
print('=== BATTING OVERVIEW ===')
print(batting[['runs','balls','fours','sixes','strike_rate']].describe().round(2))

## 3. Univariate Analysis

### 3a. Title Winners

In [ ]:
title_wins = {
    2008:'Rajasthan Royals', 2009:'Deccan Chargers', 2010:'Chennai Super Kings',
    2011:'Chennai Super Kings', 2012:'Kolkata Knight Riders', 2013:'Mumbai Indians',
    2014:'Kolkata Knight Riders', 2015:'Mumbai Indians', 2016:'Sunrisers Hyderabad',
    2017:'Mumbai Indians', 2018:'Chennai Super Kings', 2019:'Mumbai Indians',
    2020:'Mumbai Indians', 2021:'Chennai Super Kings', 2022:'Gujarat Titans',
    2023:'Chennai Super Kings'
}
tw = pd.Series(title_wins.values()).value_counts().sort_values()

fig, ax = plt.subplots(figsize=(9,5), facecolor=BG)
colors = [IPL_COLORS.get(t, BLUE) for t in tw.index]
bars = ax.barh(tw.index, tw.values, color=colors, height=0.6)
for bar, v in zip(bars, tw.values):
    ax.text(bar.get_width()+0.05, bar.get_y()+bar.get_height()/2,
            str(v), va='center', color=WHT, fontsize=11, fontweight='bold')
for sp in ax.spines.values(): sp.set_visible(False)
ax.grid(axis='x', color='#21262D', lw=0.6, ls='--')
ax.set_xlabel('Championships', color=MUT); ax.tick_params(axis='y', colors=WHT)
ax.set_title('IPL Title Winners (2008–2023)', color=WHT, fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('../charts/01_title_winners.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('\nCSK and MI dominate with 5 titles each — the two dynasties of IPL.')

### 3b. Toss Decision Distribution

In [ ]:
toss = matches['toss_decision'].value_counts()
print('Toss decision breakdown:')
for d, c in toss.items():
    print(f'  {d:8s}: {c} ({c/len(matches)*100:.1f}%)')

fig, ax = plt.subplots(figsize=(6,5), facecolor=BG)
ax.pie(toss.values, labels=toss.index, autopct='%1.1f%%',
       colors=[BLUE, ORG], startangle=90,
       textprops={'color': WHT}, wedgeprops={'edgecolor': BG, 'linewidth': 2})
ax.set_title('Toss Decision: Field vs Bat', color=WHT, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../charts/03_toss_decision.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

### 3c. Score Distribution

In [ ]:
all_scores = pd.concat([matches['team1_score'], matches['team2_score']])

fig, ax = plt.subplots(figsize=(9,5), facecolor=BG)
ax.hist(all_scores, bins=35, color=BLUE, edgecolor=BG, lw=0.4, alpha=0.85)
ax.axvline(all_scores.mean(), color=ORG, lw=1.8, ls='--', label=f'Mean: {all_scores.mean():.0f}')
ax.axvline(all_scores.median(), color=GRN, lw=1.8, ls=':', label=f'Median: {all_scores.median():.0f}')
for sp in ax.spines.values(): sp.set_visible(False)
ax.grid(axis='y', color='#21262D', lw=0.6, ls='--')
ax.set_xlabel('Total Score', color=MUT); ax.set_ylabel('Frequency', color=MUT)
ax.set_title('Distribution of Team Scores', color=WHT, fontsize=13, fontweight='bold', pad=12)
ax.legend(frameon=False, labelcolor=WHT)
plt.tight_layout()
plt.savefig('../charts/05_score_distribution.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

# Outlier detection — scores > 220 or < 100
high = all_scores[all_scores > 220]
low  = all_scores[all_scores < 100]
print(f'High-scoring innings (>220): {len(high)} ({len(high)/len(all_scores)*100:.1f}%)')
print(f'Low-scoring innings (<100) : {len(low)} ({len(low)/len(all_scores)*100:.1f}%)')

## 4. Bivariate Analysis

### 4a. Toss Advantage — Does Winning the Toss Help?

In [ ]:
matches['toss_winner_won'] = matches['toss_winner'] == matches['winner']
overall_toss_win = matches['toss_winner_won'].mean()
bat_first_win  = matches[matches['toss_decision']=='bat']['toss_winner_won'].mean()
field_first_win = matches[matches['toss_decision']=='field']['toss_winner_won'].mean()

print(f'Overall toss advantage    : {overall_toss_win:.1%}')
print(f'Win rate when chose bat   : {bat_first_win:.1%}')
print(f'Win rate when chose field : {field_first_win:.1%}')

fig, ax = plt.subplots(figsize=(7,5), facecolor=BG)
labels = ['Chose to Bat', 'Chose to Field']
vals   = [bat_first_win*100, field_first_win*100]
bars = ax.bar(labels, vals, color=[BLUE, GRN], width=0.4)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{v:.1f}%', ha='center', color=WHT, fontsize=12, fontweight='bold')
for sp in ax.spines.values(): sp.set_visible(False)
ax.grid(axis='y', color='#21262D', lw=0.6, ls='--')
ax.set_ylabel('Win %', color=MUT); ax.set_ylim(0,80)
ax.set_title('Toss Winner Win Rate by Decision', color=WHT, fontsize=13, fontweight='bold', pad=12)
plt.xticks(color=WHT)
plt.tight_layout()
plt.savefig('../charts/10_toss_advantage.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

### 4b. Pace vs Spin — Economy Rate Comparison

In [ ]:
pace = bowling[bowling['bowl_type']=='Pace']['economy']
spin = bowling[bowling['bowl_type']=='Spin']['economy']

print('Economy Rate Summary:')
print(f'  Pace — Mean: {pace.mean():.2f}, Median: {pace.median():.2f}, Std: {pace.std():.2f}')
print(f'  Spin — Mean: {spin.mean():.2f}, Median: {spin.median():.2f}, Std: {spin.std():.2f}')

fig, ax = plt.subplots(figsize=(8,5), facecolor=BG)
for data, color, label in [(pace, BLUE, 'Pace'), (spin, ORG, 'Spin')]:
    ax.hist(data, bins=25, alpha=0.65, label=label, color=color, edgecolor=BG, lw=0.4)
    ax.axvline(data.mean(), color=color, lw=1.5, ls='--')
for sp in ax.spines.values(): sp.set_visible(False)
ax.grid(axis='y', color='#21262D', lw=0.6, ls='--')
ax.set_xlabel('Economy Rate', color=MUT); ax.set_ylabel('Frequency', color=MUT)
ax.set_title('Economy Rate: Pace vs Spin', color=WHT, fontsize=13, fontweight='bold', pad=12)
ax.legend(frameon=False, labelcolor=WHT)
plt.tight_layout()
plt.savefig('../charts/09_economy_pace_spin.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

## 5. Trend Analysis

### 5a. Score Inflation Over Seasons

In [ ]:
avg_score = matches.groupby('season')['team1_score'].mean()
print('Avg first-innings score by season:')
print(avg_score.round(1).to_string())

fig, ax = plt.subplots(figsize=(10,5), facecolor=BG)
ax.plot(avg_score.index, avg_score.values, color=YLW, lw=2.5,
        marker='o', ms=6, markerfacecolor=WHT, markeredgecolor=YLW)
ax.fill_between(avg_score.index, avg_score.values, alpha=0.12, color=YLW)
for x,y in zip(avg_score.index, avg_score.values):
    ax.text(x, y+0.8, f'{y:.0f}', ha='center', color=MUT, fontsize=8)
for sp in ax.spines.values(): sp.set_visible(False)
ax.grid(axis='y', color='#21262D', lw=0.6, ls='--')
ax.set_xlabel('Season', color=MUT); ax.set_ylabel('Avg Score', color=MUT)
ax.set_title('Average First-Innings Score by Season', color=WHT, fontsize=13, fontweight='bold', pad=12)
ax.set_xticks(avg_score.index)
plt.xticks(rotation=30, color=WHT)
plt.tight_layout()
plt.savefig('../charts/04_avg_score_trend.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

### 5b. Strike Rate Evolution by Season

In [ ]:
sr = batting[batting['balls']>=5].groupby('season')['strike_rate'].mean()

fig, ax = plt.subplots(figsize=(10,5), facecolor=BG)
ax.plot(sr.index, sr.values, color=GRN, lw=2.5,
        marker='s', ms=6, markerfacecolor=WHT, markeredgecolor=GRN)
ax.fill_between(sr.index, sr.values, alpha=0.1, color=GRN)
for x,y in zip(sr.index, sr.values):
    ax.text(x, y+0.4, f'{y:.0f}', ha='center', color=MUT, fontsize=8)
for sp in ax.spines.values(): sp.set_visible(False)
ax.grid(axis='y', color='#21262D', lw=0.6, ls='--')
ax.set_xlabel('Season', color=MUT); ax.set_ylabel('Avg Strike Rate', color=MUT)
ax.set_title('Batting Strike Rate Trend by Season', color=WHT, fontsize=13, fontweight='bold', pad=12)
ax.set_xticks(sr.index)
plt.xticks(rotation=30, color=WHT)
plt.tight_layout()
plt.savefig('../charts/08_strike_rate_trend.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

## 6. Correlation Analysis

In [ ]:
bat_corr = batting[['runs','balls','fours','sixes','strike_rate']].corr()
print('Correlation matrix (batting):')
print(bat_corr.round(2))

import numpy as np
mask = np.zeros_like(bat_corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True

fig, ax = plt.subplots(figsize=(7,6), facecolor=BG)
sns.heatmap(bat_corr, ax=ax, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, linewidths=0.5, linecolor=BG,
            annot_kws={'size':11,'color':'#0D1117','fontweight':'bold'},
            cbar_kws={'shrink':0.8})
ax.set_facecolor(SRF)
ax.set_title('Batting Stats Correlation Matrix', color=WHT, fontsize=13, fontweight='bold', pad=12)
ax.tick_params(colors=WHT)
for sp in ax.spines.values(): sp.set_visible(False)
plt.tight_layout()
plt.savefig('../charts/11_correlation_heatmap.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

print('\nKey findings:')
print('  Runs ↔ Balls    : 1.00 — more balls = more runs (expected)')
print('  Runs ↔ Fours    : 0.79 — boundary hitting drives total runs')
print('  Runs ↔ SR       : 0.40 — strike rate has moderate impact')
print('  Sixes ↔ SR      : 0.16 — six hitters do not always have highest SR')

## 7. Outlier Detection

In [ ]:
# Top individual innings
print('=== HIGHEST INDIVIDUAL SCORES ===')
print(batting.nlargest(10, 'runs')[['batsman','team','season','runs','balls','sixes']].to_string(index=False))

print('\n=== LOWEST TEAM SCORES (OUTLIERS) ===')
all_sc = pd.concat([matches[['season','team1','team1_score']].rename(columns={'team1':'team','team1_score':'score'}),
                    matches[['season','team2','team2_score']].rename(columns={'team2':'team','team2_score':'score'})])
print(all_sc.nsmallest(5, 'score').to_string(index=False))

# IQR-based outlier detection on scores
Q1 = all_sc['score'].quantile(0.25)
Q3 = all_sc['score'].quantile(0.75)
IQR = Q3 - Q1
outliers_high = all_sc[all_sc['score'] > Q3 + 1.5*IQR]
outliers_low  = all_sc[all_sc['score'] < Q1 - 1.5*IQR]
print(f'\nIQR range        : {Q1:.0f} – {Q3:.0f}')
print(f'High outliers (>{Q3+1.5*IQR:.0f}) : {len(outliers_high)} innings')
print(f'Low  outliers (<{Q1-1.5*IQR:.0f})  : {len(outliers_low)} innings')

## 8. Key Findings Summary

| # | Finding | Insight |
|---|---------|----------|
| 1 | **CSK & MI dominate** with 5 titles each | Consistent squad building & retention strategy |
| 2 | **~65% of captains choose to field** after winning toss | Chasing is the preferred strategy in T20 |
| 3 | **Average scores have risen** from ~160 to ~170+ since 2018 | Better pitches, power-hitting evolution |
| 4 | **Fielding-first teams win more often** than batting-first | Dew factor + knowing the target helps chasers |
| 5 | **Fours drive run-scoring** more than sixes (correlation 0.79 vs 0.54) | Boundary placement > brute power |
| 6 | **Spin and Pace have similar economy rates** | Match conditions matter more than bowler type |
| 7 | **Strike rate weakly correlates with total runs** (0.40) | Accumulation + timing beats pure aggression |
| 8 | **~2% of innings score above 220** — rare but increasingly common | T20 format keeps evolving toward higher scores |


---
**Author:** Dheeraj Kandpal | **GitHub:** [DheerajKandpal](https://github.com/DheerajKandpal) | **Email:** dheeraj.kandpal@surepass.io
